# Sleep Health Statistical Analysis



1. Do male and female participants differ in stress level?

2. Do participants with and without sleep disorders differ in sleep duration?

3. Do insomnia and sleep apnea participants differ in physical activity level?

Dataset source: [Sleep Health and Lifestyle Dataset](https://www.kaggle.com/datasets/uom190346a/sleep-health-and-lifestyle-dataset)

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

df = pd.read_csv("../data/sleep.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())

# cleaning / preparation

df["Sleep Disorder Filled"] = df["Sleep Disorder"].fillna("None")

# Create binary sleep disorder column:
# No  = no sleep disorder
# Yes = has Insomnia or Sleep Apnea
df["Has Sleep Disorder"] = np.where(
    df["Sleep Disorder"].isna(),
    "No",
    "Yes"
)

print("\nSleep Disorder Filled counts:")
print(df["Sleep Disorder Filled"].value_counts())

print("\nHas Sleep Disorder counts:")
print(df["Has Sleep Disorder"].value_counts())

# helper function for Mann-Whitney U test

def run_mann_whitney(data, group_col, group_1, group_2, outcome_col):
    """
    Runs Mann-Whitney U test between two independent groups.

    Parameters
    ----------
    data : DataFrame
        Dataset.
    group_col : str
        Categorical column with group names.
    group_1 : str
        First group name.
    group_2 : str
        Second group name.
    outcome_col : str
        Numeric or ordinal outcome column.

    Returns
    -------
    dict
        Summary results.
    """

    x = data.loc[data[group_col] == group_1, outcome_col].dropna()
    y = data.loc[data[group_col] == group_2, outcome_col].dropna()

    result = mannwhitneyu(
        x=x,
        y=y,
        alternative="two-sided",
        method="asymptotic",
        nan_policy="omit"
    )

    return {
        "Research Question": f"Does {outcome_col} differ between {group_1} and {group_2}?",
        "Grouping Column": group_col,
        "Outcome": outcome_col,
        "Group 1": group_1,
        "N1": len(x),
        "Median 1": x.median(),
        "Mean 1": x.mean(),
        "Group 2": group_2,
        "N2": len(y),
        "Median 2": y.median(),
        "Mean 2": y.mean(),
        "U Statistic": result.statistic,
        "p-value": result.pvalue,
        "Significant": result.pvalue < 0.05
    }

# Scenario 1

# Question:
# Do male and female participants differ in stress level?

scenario_1 = run_mann_whitney(
    data=df,
    group_col="Gender",
    group_1="Male",
    group_2="Female",
    outcome_col="Stress Level"
)

print("\n" + "=" * 80)
print("SCENARIO 1: Gender vs Stress Level")
print("=" * 80)

for key, value in scenario_1.items():
    print(f"{key}: {value}")


# Scenario 2

# Question:
# Do participants with and without sleep disorders differ in sleep duration?

scenario_2 = run_mann_whitney(
    data=df,
    group_col="Has Sleep Disorder",
    group_1="No",
    group_2="Yes",
    outcome_col="Sleep Duration"
)

print("\n" + "=" * 80)
print("SCENARIO 2: Sleep Disorder Status vs Sleep Duration")
print("=" * 80)

for key, value in scenario_2.items():
    print(f"{key}: {value}")


# Scenario 3

# Question:
# Do insomnia and sleep apnea participants differ in physical activity level?

scenario_3 = run_mann_whitney(
    data=df,
    group_col="Sleep Disorder Filled",
    group_1="Insomnia",
    group_2="Sleep Apnea",
    outcome_col="Physical Activity Level"
)

print("\n" + "=" * 80)
print("SCENARIO 3: Insomnia vs Sleep Apnea on Physical Activity Level")
print("=" * 80)

for key, value in scenario_3.items():
    print(f"{key}: {value}")


# combine all results into one table

results_df = pd.DataFrame([scenario_1, scenario_2, scenario_3])

print("\n" + "=" * 80)
print("FINAL RESULTS TABLE")
print("=" * 80)

print(results_df)


results_clean = results_df.copy()

numeric_cols = [
    "Median 1", "Mean 1",
    "Median 2", "Mean 2",
    "U Statistic", "p-value"
]

# results

results_clean[numeric_cols] = results_clean[numeric_cols]

print("\n" + "=" * 80)
print("CLEAN FINAL RESULTS TABLE")
print("=" * 80)

print(results_clean)


# interpretation

print("\n" + "=" * 80)
print("INTERPRETATIONS")
print("=" * 80)

alpha = 0.05

for i, row in results_clean.iterrows():
    print(f"\nScenario {i + 1}:")
    print(row["Research Question"])
    print(f"{row['Group 1']} median = {row['Median 1']}")
    print(f"{row['Group 2']} median = {row['Median 2']}")
    print(f"U = {row['U Statistic']}")
    print(f"p-value = {row['p-value']}")

    if row["p-value"] < alpha:
        print("Conclusion: Significant difference between the two groups.")
    else:
        print("Conclusion: No significant difference between the two groups.")

Dataset shape: (374, 13)

Columns:
Index(['Person ID', 'Gender', 'Age', 'Occupation', 'Sleep Duration',
       'Quality of Sleep', 'Physical Activity Level', 'Stress Level',
       'BMI Category', 'Blood Pressure', 'Heart Rate', 'Daily Steps',
       'Sleep Disorder'],
      dtype='object')

First 5 rows:
   Person ID Gender  Age            Occupation  Sleep Duration  \
0          1   Male   27     Software Engineer             6.1   
1          2   Male   28                Doctor             6.2   
2          3   Male   28                Doctor             6.2   
3          4   Male   28  Sales Representative             5.9   
4          5   Male   28  Sales Representative             5.9   

   Quality of Sleep  Physical Activity Level  Stress Level BMI Category  \
0                 6                       42             6   Overweight   
1                 6                       60             8       Normal   
2                 6                       60             8       Normal